In [3]:
# ============================================================================
# COMPLETE CONVERSATION EXTRACTION - READY TO RUN
# Copy this entire script into Google Colab and run it!
# ============================================================================

import pandas as pd
import numpy as np
import re
from typing import List
from tqdm import tqdm
import os

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ============================================================================
# FIXED MENTION EXTRACTION (no spaces in usernames)
# ============================================================================

MENTION_RE = re.compile(r'@([A-Za-z0-9_\-]+)')

def extract_mentions_list(text: str) -> List[str]:
    """Extract @mentions. FIXED: '@Leah Harris W!!' now extracts just 'leah'"""
    if pd.isna(text):
        return []

    raw_mentions = MENTION_RE.findall(str(text))
    cleaned = []
    for m in raw_mentions:
        m = m.strip().strip(".,!?:;").lower()
        if m:
            cleaned.append(m)

    seen = set()
    unique_mentions = []
    for m in cleaned:
        if m not in seen:
            unique_mentions.append(m)
            seen.add(m)

    return unique_mentions

# ============================================================================
# CONVERSATION EXTRACTION - PROPERLY TYPED (no file size bloat!)
# ============================================================================

def build_conversations_by_target(
    df_event: pd.DataFrame,
    datetime_col: str,
    lookback_window: str = "30min",
    response_window: str = "15min",
    min_messages: int = 2,
    exclude_streamer_mentions: bool = False,
    streamer_aliases: List[str] = None,
    starting_conv_id: int = 0,  # NEW: Accept starting ID
) -> tuple:  # NEW: Return tuple (dataframe, next_id)
    """
    Extract conversations as threads anchored to mentioned users.
    Each mentioned user gets their own conversation thread.

    Returns: (df_with_conversations, next_available_conv_id)
    """

    df = df_event.copy()
    df = df.dropna(subset=["username", datetime_col, "comment_text"])

    if df.empty:
        df["conversation_id"] = pd.Series(dtype='float64')
        df["conversation_type"] = pd.Series(dtype='object')
        df["num_participants"] = pd.Series(dtype='float64')
        df["conversation_depth"] = pd.Series(dtype='float64')
        df["is_root_comment"] = pd.Series(dtype='int64')
        df["conversation_has_root"] = pd.Series(dtype='int64')
        df["root_user"] = pd.Series(dtype='object')
        return df, starting_conv_id  # Return unchanged ID

    df = df.sort_values(datetime_col).reset_index(drop=True)
    df["_user_norm"] = df["username"].astype(str).str.strip().str.lower()
    df["_mentions_list"] = df["comment_text"].apply(extract_mentions_list)

    if exclude_streamer_mentions and streamer_aliases:
        streamer_set = {alias.lower() for alias in streamer_aliases}
        df["_mentions_list"] = df["_mentions_list"].apply(
            lambda mentions: [m for m in mentions if m not in streamer_set]
        )

    conversations = []
    conv_id = starting_conv_id  # START from passed ID, not 0!

    all_mentioned_users = set()
    for mentions_list in df["_mentions_list"]:
        all_mentioned_users.update(mentions_list)

    for target_user in all_mentioned_users:
        messages_mentioning_target = df[
            df["_mentions_list"].apply(lambda m: target_user in m)
        ].copy()

        if messages_mentioning_target.empty:
            continue

        first_mention_time = messages_mentioning_target[datetime_col].min()
        last_mention_time = first_mention_time + pd.Timedelta(response_window)

        thread_messages = messages_mentioning_target[
            messages_mentioning_target[datetime_col] <= last_mention_time
        ]

        if len(thread_messages) < min_messages:
            continue

        lookback_start = first_mention_time - pd.Timedelta(lookback_window)

        potential_roots = df[
            (df["_user_norm"] == target_user) &
            (df[datetime_col] < first_mention_time) &
            (df[datetime_col] >= lookback_start)
        ]

        root_idx = None
        has_root = False

        if not potential_roots.empty:
            root_idx = potential_roots[datetime_col].idxmax()
            has_root = True

        message_indices = set(thread_messages.index)
        if root_idx is not None:
            message_indices.add(root_idx)

        if len(message_indices) >= min_messages or (has_root and len(message_indices) >= min_messages - 1):
            conversations.append({
                'conversation_id': conv_id,
                'target_user': target_user,
                'message_indices': message_indices,
                'root_idx': root_idx,
                'has_root': has_root,
            })
            conv_id += 1  # Increment GLOBAL counter

    # PRE-ALLOCATE with correct types (prevents FutureWarnings and file bloat!)
    df["conversation_id"] = pd.Series(np.nan, index=df.index, dtype='float64')
    df["conversation_type"] = pd.Series("", index=df.index, dtype='object')
    df["num_participants"] = pd.Series(np.nan, index=df.index, dtype='float64')
    df["conversation_depth"] = pd.Series(np.nan, index=df.index, dtype='float64')
    df["is_root_comment"] = pd.Series(0, index=df.index, dtype='int64')
    df["conversation_has_root"] = pd.Series(0, index=df.index, dtype='int64')
    df["root_user"] = pd.Series("", index=df.index, dtype='object')

    for conv in conversations:
        indices = list(conv['message_indices'])

        conv_messages = df.loc[indices]
        num_participants = conv_messages["_user_norm"].nunique()
        conversation_depth = len(indices)

        if num_participants == 1:
            conv_type = "self_thread"
        elif num_participants == 2:
            conv_type = "dyadic_exchange"
        else:
            conv_type = "multi_party_thread"

        # Use .loc with explicit types (no warnings!)
        df.loc[indices, "conversation_id"] = float(conv['conversation_id'])
        df.loc[indices, "num_participants"] = float(num_participants)
        df.loc[indices, "conversation_depth"] = float(conversation_depth)
        df.loc[indices, "conversation_type"] = conv_type
        df.loc[indices, "conversation_has_root"] = 1 if conv['has_root'] else 0
        df.loc[indices, "root_user"] = conv['target_user']

        if conv['root_idx'] is not None:
            df.loc[conv['root_idx'], "is_root_comment"] = 1

    df = df.drop(columns=["_user_norm", "_mentions_list"])

    return df, conv_id  # Return next available ID

# ============================================================================
# MAIN PROCESSING
# ============================================================================

INPUT_PATH = '/content/drive/MyDrive/**CORE PROJECTS - Julie/[PROJECTS PAPERS]/ACTIVE IN-PROGRESS STUDIES/[Complete] [DS3] Trust Behaviors in Collective Chat/Data/All YouTube Data/youtube_comments_combined.csv'

OUTPUT_PATH = '/content/drive/MyDrive/**CORE PROJECTS - Julie/[PROJECTS PAPERS]/ACTIVE IN-PROGRESS STUDIES/[Complete] [DS3] Trust Behaviors in Collective Chat/Data/All YouTube Data/youtube_comments_with_conversations_FINAL.csv'

print("="*80)
print("CONVERSATION EXTRACTION - COMPLETE VERSION")
print("="*80)

# Load data
print(f"\nLoading: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH)
print(f"✓ Loaded {len(df):,} comments")

# Check original file size
original_size_mb = os.path.getsize(INPUT_PATH) / (1024*1024)
print(f"✓ Original file size: {original_size_mb:.1f} MB")

# Find datetime column
datetime_col_name = None
for col in ['comment_timestamp', 'comment_datetime', 'timestamp', 'datetime']:
    if col in df.columns:
        datetime_col_name = col
        print(f"✓ Using datetime column: '{col}'")
        break

if not datetime_col_name:
    print("ERROR: No datetime column found!")
    raise ValueError("No datetime column")

df[datetime_col_name] = pd.to_datetime(df[datetime_col_name])

# Process by event with GLOBAL conversation IDs
df['event_key'] = df['creator'] + '_' + df['event_date'].astype(str)
event_keys = df['event_key'].unique()

print(f"\n✓ Found {len(event_keys)} (creator, event_date) combinations")
print(f"✓ Date range: {df['event_date'].min()} to {df['event_date'].max()}")
print(f"\nProcessing conversations...")

results = []
global_conv_id = 0  # GLOBAL counter across all events!

for event_key in tqdm(event_keys, desc="Processing events"):
    df_event = df[df['event_key'] == event_key].copy()

    df_event_with_convs, next_conv_id = build_conversations_by_target(
        df_event=df_event,
        datetime_col=datetime_col_name,
        lookback_window='30min',
        response_window='15min',
        min_messages=2,
        exclude_streamer_mentions=True,
        streamer_aliases=['max', 'MaxVelocity', 'ryanmaue', 'RyanMaue', 'ryan', 'ryanhallyall'],
        starting_conv_id=global_conv_id  # Pass the global ID!
    )

    global_conv_id = next_conv_id  # Update for next event
    results.append(df_event_with_convs)

print("\n✓ Combining results...")
df_final = pd.concat(results, ignore_index=True)
df_final = df_final.drop(columns=['event_key'])

# Summary stats
messages_in_convs = df_final['conversation_id'].notna().sum()
unique_convs = df_final['conversation_id'].nunique()

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
print(f"Total messages: {len(df_final):,}")
print(f"Messages in conversations: {messages_in_convs:,} ({messages_in_convs/len(df_final)*100:.1f}%)")
print(f"Unique conversations found: {unique_convs:,}")
print(f"  (This is the REAL count - previously all had same ID!)")

if unique_convs > 0:
    convs_with_roots = df_final[df_final['conversation_id'].notna()].groupby('conversation_id')['conversation_has_root'].first()
    root_rate = convs_with_roots.sum() / len(convs_with_roots)
    print(f"\nConversations with roots: {convs_with_roots.sum():,} ({root_rate*100:.1f}%)")

    conv_types = df_final[df_final['conversation_id'].notna()].groupby('conversation_id')['conversation_type'].first().value_counts()
    print(f"\nConversation types:")
    for ctype, count in conv_types.items():
        print(f"  {ctype}: {count:,} ({count/unique_convs*100:.1f}%)")

# Save
print(f"\n" + "="*80)
print("SAVING")
print("="*80)
print(f"Saving to: {OUTPUT_PATH}")
df_final.to_csv(OUTPUT_PATH, index=False)

# Check output file size
output_size_mb = os.path.getsize(OUTPUT_PATH) / (1024*1024)
print(f"✓ Saved {len(df_final):,} rows")
print(f"✓ Output file size: {output_size_mb:.1f} MB")
print(f"✓ Size ratio: {output_size_mb/original_size_mb:.2f}x original")

if output_size_mb/original_size_mb > 3.0:
    print(f"⚠ WARNING: File seems large (expected ~2x, got {output_size_mb/original_size_mb:.1f}x)")
else:
    print(f"✓ File size looks good!")

print("\n" + "="*80)
print("✓✓✓ COMPLETE! ✓✓✓")
print("="*80)
print(f"\nYour file is ready:")
print(f"  {OUTPUT_PATH}")
print(f"\nNew columns:")
print(f"  • conversation_id")
print(f"  • root_user")
print(f"  • is_root_comment")
print(f"  • conversation_has_root")
print(f"  • num_participants")
print(f"  • conversation_depth")
print(f"  • conversation_type")
print("="*80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CONVERSATION EXTRACTION - COMPLETE VERSION

Loading: /content/drive/MyDrive/**CORE PROJECTS - Julie/[PROJECTS PAPERS]/ACTIVE IN-PROGRESS STUDIES/[Complete] [DS3] Trust Behaviors in Collective Chat/Data/All YouTube Data/youtube_comments_combined.csv
✓ Loaded 1,116,371 comments
✓ Original file size: 148.3 MB
✓ Using datetime column: 'comment_timestamp'

✓ Found 34 (creator, event_date) combinations
✓ Date range: 2025-03-14 to 2025-06-21

Processing conversations...


Processing events: 100%|██████████| 34/34 [04:44<00:00,  8.38s/it]



✓ Combining results...

RESULTS SUMMARY
Total messages: 1,115,229
Messages in conversations: 8,112 (0.7%)
Unique conversations found: 2,812
  (This is the REAL count - previously all had same ID!)

Conversations with roots: 976 (34.7%)

Conversation types:
  dyadic_exchange: 1,261 (44.8%)
  multi_party_thread: 1,031 (36.7%)
  self_thread: 520 (18.5%)

SAVING
Saving to: /content/drive/MyDrive/**CORE PROJECTS - Julie/[PROJECTS PAPERS]/ACTIVE IN-PROGRESS STUDIES/[Complete] [DS3] Trust Behaviors in Collective Chat/Data/All YouTube Data/youtube_comments_with_conversations_FINAL.csv
✓ Saved 1,115,229 rows
✓ Output file size: 158.0 MB
✓ Size ratio: 1.07x original
✓ File size looks good!

✓✓✓ COMPLETE! ✓✓✓

Your file is ready:
  /content/drive/MyDrive/**CORE PROJECTS - Julie/[PROJECTS PAPERS]/ACTIVE IN-PROGRESS STUDIES/[Complete] [DS3] Trust Behaviors in Collective Chat/Data/All YouTube Data/youtube_comments_with_conversations_FINAL.csv

New columns:
  • conversation_id
  • root_user
  • is_r

In [5]:
# SAVE ONLY CONVERSATIONS - Much smaller file!
# This keeps only the ~8,000 messages that are IN conversations

import pandas as pd
import os

INPUT_FILE = '/content/drive/MyDrive/**CORE PROJECTS - Julie/[PROJECTS PAPERS]/ACTIVE IN-PROGRESS STUDIES/[Complete] [DS3] Trust Behaviors in Collective Chat/Data/All YouTube Data/youtube_comments_with_conversations_FINAL.csv'

OUTPUT_FILE = '/content/drive/MyDrive/**CORE PROJECTS - Julie/[PROJECTS PAPERS]/ACTIVE IN-PROGRESS STUDIES/[Complete] [DS3] Trust Behaviors in Collective Chat/Data/All YouTube Data/CONVERSATIONS_ONLY.csv'

print("="*80)
print("CREATING CONVERSATIONS-ONLY FILE")
print("="*80)

# Read in chunks to avoid memory issues
print("\nReading file in chunks...")

chunk_size = 100000
first_chunk = True
total_kept = 0

for chunk in pd.read_csv(INPUT_FILE, chunksize=chunk_size):
    # Keep only rows in conversations
    chunk_convs = chunk[chunk['conversation_id'].notna()].copy()

    if len(chunk_convs) == 0:
        continue

    # Write
    if first_chunk:
        chunk_convs.to_csv(OUTPUT_FILE, index=False, mode='w')
        first_chunk = False
    else:
        chunk_convs.to_csv(OUTPUT_FILE, index=False, mode='a', header=False)

    total_kept += len(chunk_convs)
    print(f"  Processed chunk, kept {total_kept:,} conversation messages so far...")

print(f"\n✓ Done! Kept {total_kept:,} messages in conversations")

# Check file size
output_size = os.path.getsize(OUTPUT_FILE) / (1024*1024)
print(f"✓ Output file size: {output_size:.1f} MB")

# Check cell count for Google Sheets
num_cols = pd.read_csv(OUTPUT_FILE, nrows=1).shape[1]
total_cells = total_kept * num_cols

print(f"\nGoogle Sheets check:")
print(f"  Rows: {total_kept:,}")
print(f"  Columns: {num_cols}")
print(f"  Total cells: {total_cells:,}")

if total_cells < 10_000_000:
    print(f"  ✓ FITS in Google Sheets! ({total_cells/1_000_000:.1f}M / 10M limit)")
else:
    print(f"  ✗ Still too big ({total_cells/1_000_000:.1f}M / 10M limit)")

print("\n" + "="*80)
print("DONE!")
print("="*80)
print(f"\nYour conversations-only file:")
print(f"  {OUTPUT_FILE}")
print(f"\nThis should open in Google Sheets!")
print("="*80)

CREATING CONVERSATIONS-ONLY FILE

Reading file in chunks...
  Processed chunk, kept 772 conversation messages so far...
  Processed chunk, kept 1,532 conversation messages so far...
  Processed chunk, kept 2,832 conversation messages so far...
  Processed chunk, kept 3,896 conversation messages so far...
  Processed chunk, kept 4,854 conversation messages so far...
  Processed chunk, kept 5,460 conversation messages so far...
  Processed chunk, kept 5,747 conversation messages so far...
  Processed chunk, kept 6,110 conversation messages so far...
  Processed chunk, kept 6,380 conversation messages so far...
  Processed chunk, kept 6,791 conversation messages so far...
  Processed chunk, kept 7,671 conversation messages so far...
  Processed chunk, kept 8,112 conversation messages so far...

✓ Done! Kept 8,112 messages in conversations
✓ Output file size: 1.5 MB

Google Sheets check:
  Rows: 8,112
  Columns: 14
  Total cells: 113,568
  ✓ FITS in Google Sheets! (0.1M / 10M limit)

DONE!